# Advanced Architectures for Automatic Modulation Recognition

**Students:** Lyes HADJAR, Moulay Mohamed BOUABDELLI  
**Course:** Wireless Communication Networks and Systems  
**Institution:** ENSIA - 4th Year, Semester 1 (2025/2026)

---

This notebook implements and evaluates advanced deep learning architectures for AMR, including:
- ResNet-Inspired CNN
- Bidirectional LSTM
- Hybrid CNN-LSTM
- **CNN-Transformer with Latent Attention** (Primary Novel Architecture)
- Pure Transformer
- Ensemble Methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D, GlobalMaxPooling1D,
    Flatten, Dense, Dropout, LSTM, Bidirectional, Add, Multiply,
    BatchNormalization, Activation, MultiHeadAttention, LayerNormalization,
    Concatenate, Lambda
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, precision_recall_fscore_support
)
import pickle
import os

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Parameters
NUM_CLASSES = 5
MOD_LABELS = ['BPSK', '4-PAM', '8PSK', '16QAM', '2-FSK']
DATASETS = {"Raw": "dataset", "ZF": "dataset_ZF", "MMSE": "dataset_MMSE"}
SNR_LIST = [-5, 0, 5, 10, 15, 20, 25]

def load_data(base_path, snr):
    """Load and preprocess data"""
    X = np.load(f"{base_path}/SNR_{snr}/X.npy")
    y = np.load(f"{base_path}/SNR_{snr}/y.npy")
    X = np.transpose(X, (0, 2, 1))  # (N, 1024, 2)
    X = X / (np.max(np.abs(X), axis=(1, 2), keepdims=True) + 1e-8)
    return X, y

## 1. ResNet-Inspired Deep CNN

In [ ]:
def residual_block(x, filters, kernel_size=3):
    """Residual block with skip connection"""
    shortcut = x
    
    x = Conv1D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv1D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    
    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(filters, 1, padding='same')(shortcut)
    
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def build_resnet_amr():
    """ResNet-inspired architecture for AMR"""
    inputs = Input(shape=(1024, 2))
    
    x = Conv1D(64, 7, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = residual_block(x, 64)
    x = residual_block(x, 64)
    x = MaxPooling1D(2)(x)
    
    x = residual_block(x, 128)
    x = residual_block(x, 128)
    x = MaxPooling1D(2)(x)
    
    x = residual_block(x, 256)
    x = residual_block(x, 256)
    x = GlobalAveragePooling1D()(x)
    
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='ResNet_AMR')
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

## 2. Bidirectional LSTM

In [ ]:
def build_lstm_amr():
    """Bidirectional LSTM for temporal modeling"""
    model = Sequential([
        Input(shape=(1024, 2)),
        
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.3),
        
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.3),
        
        Bidirectional(LSTM(64)),
        Dropout(0.3),
        
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax')
    ], name='BiLSTM_AMR')
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

## 3. Hybrid CNN-LSTM

In [ ]:
def build_cnn_lstm():
    """Hybrid CNN-LSTM combining local and temporal features"""
    model = Sequential([
        Input(shape=(1024, 2)),
        
        # CNN for local feature extraction
        Conv1D(64, 7, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        
        Conv1D(128, 5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        
        Conv1D(256, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        
        # LSTM for temporal dependencies
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(64)),
        Dropout(0.3),
        
        # Classification
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax')
    ], name='CNN_LSTM')
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

## 4. CNN-Transformer with Latent Attention (Novel Architecture)

This is the **primary proposed architecture** combining:
- CNN feature extraction for local patterns
- Transformer blocks with multi-head attention for global context
- Latent representation layer for discriminative embedding
- Attention-weighted pooling for interpretability

In [ ]:
def positional_encoding(length, depth):
    """Generate positional encodings for transformer"""
    positions = np.arange(length)[:, np.newaxis]
    depths = np.arange(depth)[np.newaxis, :] / depth
    
    angle_rates = 1 / (10000**depths)
    angle_rads = positions * angle_rates
    
    pos_encoding = np.concatenate([
        np.sin(angle_rads[:, 0::2]),
        np.cos(angle_rads[:, 1::2])
    ], axis=-1)
    
    return tf.cast(pos_encoding, dtype=tf.float32)

class AddPositionalEncoding(tf.keras.layers.Layer):
    """Add positional encoding to inputs"""
    def __init__(self, **kwargs):
        super(AddPositionalEncoding, self).__init__(**kwargs)
    
    def call(self, inputs):
        length = tf.shape(inputs)[1]
        depth = tf.shape(inputs)[2]
        pos_enc = positional_encoding(length, depth)
        return inputs + pos_enc[tf.newaxis, :, :]

def transformer_block(x, num_heads, key_dim, ff_dim, dropout=0.1):
    """Single transformer encoder block"""
    # Multi-head attention
    attn_output = MultiHeadAttention(
        num_heads=num_heads, 
        key_dim=key_dim,
        dropout=dropout
    )(x, x)
    attn_output = Dropout(dropout)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(x + attn_output)
    
    # Feed-forward network
    ffn_output = Dense(ff_dim, activation='relu')(out1)
    ffn_output = Dropout(dropout)(ffn_output)
    ffn_output = Dense(x.shape[-1])(ffn_output)
    
    return LayerNormalization(epsilon=1e-6)(out1 + ffn_output)

In [ ]:
def build_cnn_transformer_latent():
    """
    CNN-Transformer architecture with latent representation
    
    Flow:
    1. CNN Feature Extraction - Local patterns
    2. Positional Encoding - Add position information
    3. Transformer Blocks - Global context with attention
    4. Latent Representation - Compressed discriminative space
    5. Attention Pooling - Weighted feature aggregation
    6. Classification Head (FFN) - Final prediction
    """
    inputs = Input(shape=(1024, 2), name='input_signal')
    
    # ==========================================
    # CNN Feature Extractor
    # ==========================================
    x = Conv1D(64, 7, padding='same', activation='relu', name='conv1')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)  # 1024 -> 512
    
    x = Conv1D(128, 5, padding='same', activation='relu', name='conv2')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)  # 512 -> 256
    
    x = Conv1D(256, 3, padding='same', activation='relu', name='conv3')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)  # 256 -> 128
    
    # Project to transformer dimension
    cnn_features = Conv1D(256, 1, activation='relu',
                          name='feature_projection')(x)
    
    # ==========================================
    # Add Positional Encoding
    # ==========================================
    x = AddPositionalEncoding()(cnn_features)
    
    # ==========================================
    # Transformer Encoder Blocks
    # ==========================================
    for i in range(3):
        x = transformer_block(
            x, 
            num_heads=8, 
            key_dim=32,
            ff_dim=512,
            dropout=0.1
        )
    
    transformer_features = x
    
    # ==========================================
    # Latent Representation Layer
    # ==========================================
    latent_dim = 128
    
    global_features = GlobalAveragePooling1D()(transformer_features)
    latent_repr = Dense(latent_dim, activation='relu',
                       name='latent_representation')(global_features)
    latent_repr = BatchNormalization()(latent_repr)
    latent_repr = Dropout(0.3)(latent_repr)
    
    # ==========================================
    # Attention-based Weighted Pooling
    # ==========================================
    attention_weights = Dense(1, activation='tanh')(transformer_features)
    attention_weights = tf.keras.layers.Softmax(axis=1)(attention_weights)
    
    attended_features = Multiply()([transformer_features, attention_weights])
    attended_pooled = tf.reduce_sum(attended_features, axis=1)
    
    # ==========================================
    # Feature Fusion
    # ==========================================
    fused_features = Concatenate()([latent_repr, attended_pooled])
    
    # ==========================================
    # Classification Head (FFN)
    # ==========================================
    x = Dense(256, activation='relu', name='ffn_layer1')(fused_features)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    
    x = Dense(128, activation='relu', name='ffn_layer2')(x)
    x = Dropout(0.3)(x)
    
    outputs = Dense(NUM_CLASSES, activation='softmax',
                   name='classification')(x)
    
    model = Model(inputs=inputs, outputs=outputs,
                 name='CNN_Transformer_Latent')
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

## 5. Pure Transformer

In [ ]:
def build_transformer_amr():
    """Pure transformer architecture"""
    inputs = Input(shape=(1024, 2))
    
    # Positional embedding
    x = Dense(128)(inputs)
    x = AddPositionalEncoding()(x)
    
    # Transformer blocks
    for _ in range(4):
        x = transformer_block(x, num_heads=4, key_dim=32,
                            ff_dim=256, dropout=0.1)
    
    x = GlobalAveragePooling1D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='Transformer_AMR')
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

## 6. Model Comparison Framework

In [ ]:
# Define all models to compare
MODELS = {
    'ResNet': build_resnet_amr,
    'BiLSTM': build_lstm_amr,
    'CNN-LSTM': build_cnn_lstm,
    'CNN-Transformer': build_cnn_transformer_latent,  # Primary architecture
    'Transformer': build_transformer_amr
}

# Display model summaries
for name, build_fn in MODELS.items():
    print(f"\n{'='*60}")
    print(f" {name} Architecture")
    print('='*60)
    model = build_fn()
    model.summary()
    print(f"Total parameters: {model.count_params():,}")

## 7. Training Function with Callbacks

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, model_name, epochs=30):
    """Train model with proper callbacks"""
    
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            f'best_{model_name}.h5',
            save_best_only=True,
            monitor='val_accuracy',
            mode='max',
            verbose=1
        )
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=64,
        callbacks=callbacks,
        verbose=1
    )
    
    return history

## 8. Evaluation and Comparison

Train and evaluate all models on MMSE-equalized data (recommended for best performance).

In [ ]:
# Choose dataset (MMSE recommended)
DATASET_NAME = "MMSE"
DATASET_PATH = DATASETS[DATASET_NAME]

# Choose SNR for training (recommend 10 or 15 dB for balanced performance)
TRAIN_SNR = 15

print(f"Training on {DATASET_NAME} dataset at SNR = {TRAIN_SNR} dB")

# Load data
X, y = load_data(DATASET_PATH, TRAIN_SNR)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Train all models
trained_models = {}
histories = {}

for name, build_fn in MODELS.items():
    print(f"\n\n{'='*70}")
    print(f" Training {name}")
    print('='*70)
    
    model = build_fn()
    history = train_model(model, X_train, y_train, X_val, y_val,
                         model_name=f"{name}_{DATASET_NAME}_SNR{TRAIN_SNR}")
    
    trained_models[name] = model
    histories[name] = history.history
    
    # Evaluate on test set
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\n{name} Test Accuracy: {test_acc*100:.2f}%")

## 9. Training Dynamics Visualization

In [ ]:
# Plot training curves for all models
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, history) in enumerate(histories.items()):
    if idx >= 6:
        break
    
    ax = axes[idx]
    
    # Plot accuracy
    ax.plot(history['accuracy'], label='Train Acc', linewidth=2)
    ax.plot(history['val_accuracy'], label='Val Acc', linewidth=2)
    ax.set_title(f'{name} - Training Dynamics', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Remove extra subplots
for idx in range(len(histories), 6):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('training_dynamics_all_models.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Cross-SNR Evaluation

Evaluate all models across all SNR levels to assess robustness.

In [ ]:
# Evaluate all models across all SNRs
snr_results = {name: [] for name in MODELS.keys()}

for snr in SNR_LIST:
    print(f"\nEvaluating at SNR = {snr} dB")
    X_snr, y_snr = load_data(DATASET_PATH, snr)
    
    for name, model in trained_models.items():
        y_pred = np.argmax(model.predict(X_snr, verbose=0), axis=1)
        acc = accuracy_score(y_snr, y_pred)
        snr_results[name].append(acc)
        print(f"  {name}: {acc*100:.2f}%")

# Plot SNR comparison
plt.figure(figsize=(12, 7))
for name, accs in snr_results.items():
    linestyle = '--' if name == 'CNN-Transformer' else '-'
    linewidth = 3 if name == 'CNN-Transformer' else 2
    plt.plot(SNR_LIST, accs, marker='o', linestyle=linestyle,
            linewidth=linewidth, label=name, markersize=8)

plt.xlabel('SNR (dB)', fontsize=14)
plt.ylabel('Accuracy', fontsize=14)
plt.title(f'Model Comparison Across SNR Levels ({DATASET_NAME} Dataset)',
         fontsize=16, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12, loc='lower right')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig('snr_comparison_all_models.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Detailed Performance Analysis (CNN-Transformer)

In [ ]:
# Focus on our novel CNN-Transformer architecture
model_cnn_transformer = trained_models['CNN-Transformer']

# Generate confusion matrices at different SNRs
test_snrs = [-5, 10, 25]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, snr in enumerate(test_snrs):
    X_snr, y_snr = load_data(DATASET_PATH, snr)
    y_pred = np.argmax(model_cnn_transformer.predict(X_snr, verbose=0), axis=1)
    
    cm = confusion_matrix(y_snr, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=MOD_LABELS)
    disp.plot(ax=axes[idx], cmap='Blues', values_format='d')
    
    acc = accuracy_score(y_snr, y_pred)
    axes[idx].set_title(f'CNN-Transformer\nSNR = {snr} dB (Acc: {acc*100:.1f}%)',
                       fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('cnn_transformer_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Per-modulation analysis for CNN-Transformer
snr_test = 15
X_snr, y_snr = load_data(DATASET_PATH, snr_test)
y_pred = np.argmax(model_cnn_transformer.predict(X_snr, verbose=0), axis=1)

print(f"\nCNN-Transformer Performance at SNR = {snr_test} dB\n")
print(classification_report(y_snr, y_pred, target_names=MOD_LABELS, digits=4))

## 12. Save Results

In [ ]:
# Save all results
results_data = {
    'snr_results': snr_results,
    'histories': histories,
    'dataset': DATASET_NAME,
    'train_snr': TRAIN_SNR,
    'models': list(MODELS.keys())
}

with open('advanced_architectures_results.pkl', 'wb') as f:
    pickle.dump(results_data, f)

print("\nResults saved to advanced_architectures_results.pkl")

# Save best models
for name, model in trained_models.items():
    model.save(f'final_{name}_{DATASET_NAME}.h5')
    print(f"Saved model: final_{name}_{DATASET_NAME}.h5")

## 13. Summary Table

In [ ]:
import pandas as pd

# Create summary table
summary_data = []
for name in MODELS.keys():
    model = trained_models[name]
    params = model.count_params()
    
    # Accuracy at different SNRs
    acc_low = snr_results[name][0] * 100  # -5 dB
    acc_mid = snr_results[name][3] * 100  # 10 dB
    acc_high = snr_results[name][-1] * 100  # 25 dB
    
    summary_data.append({
        'Model': name,
        'Parameters': f'{params:,}',
        'Acc @ -5dB': f'{acc_low:.2f}%',
        'Acc @ 10dB': f'{acc_mid:.2f}%',
        'Acc @ 25dB': f'{acc_high:.2f}%'
    })

df_summary = pd.DataFrame(summary_data)
print("\n" + "="*80)
print(" Model Comparison Summary")
print("="*80)
print(df_summary.to_string(index=False))
print("="*80)

# Save to CSV
df_summary.to_csv('model_comparison_summary.csv', index=False)
print("\nSummary saved to model_comparison_summary.csv")


---

**Students:** Lyes HADJAR, Moulay Mohamed BOUABDELLI  
**Institution:** ENSIA - National School of Artificial Intelligence